# Amaliy mashgʻulot 7. Matnlarni tasniflash: Naive Bayes, SVM va tonallik tahlili

7-laboratoriyada sharhdagi har bir soʻz ijobiy yoki salbiy sinf uchun qanday «ovoz berishini», Laplas silliqlashi nol ehtimollardan qanday qutqarishini, inkor qanday taʼsir qilishini va SVM ajratuvchi yoʻlakni qanday qidirishini koʻrdingiz. Bu yerda hammasini 60 ta qisqa sharhda kodda takrorlaymiz:

1. multinomial sodda Bayes ni noldan yozamiz (ehtimollar logarifmlari, Laplas silliqlashi) va `sklearn.naive_bayes.MultinomialNB` bilan solishtiramiz;
2. qaysi soʻzlar sharhni qaysi tomonga tortishini tahlil qilamiz va modelga inkorni tushunishni oʻrgatamiz («yaxshi emas» → `EMAS_yaxshi`);
3. chiziqli SVM ni hinge loss boʻyicha gradient tushish bilan `numpy` da oʻqitamiz, soʻng scikit-learn dagi `LinearSVC` va `LogisticRegression`;
4. modellarni baholaymiz: xatolar matritsasi, aniqlik va toʻliqlik, kross-validatsiya.

*«NLP algoritmlari» kursining 7-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Maʼlumotlar: 60 ta sharh

Filmlar, kafelar, kitoblar va texnika haqida 30 ta ijobiy va 30 ta salbiy sharh. Sharhlar ataylab juft qilib yozilgan: bir xil mavzular («aktyorlar», «sifati», «xizmat») ikkala sinfda ham uchraydi, shuning uchun model baholovchi soʻzlarga qarashi kerak boʻladi. Har bir sinfning oxirgi 6 ta sharhini test uchun ajratamiz: model ularni oʻqitishda koʻrmaydi.

*Maslahat: oʻz maʼlumotlaringizni chapdagi «Files» paneli orqali yuklashingiz mumkin (masalan, `text,label` ustunli `reviews.csv`) va `pd.read_csv("reviews.csv")` bilan oʻqing.*

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pos = [
    "Kino juda yoqdi, syujeti ajoyib",
    "Ajoyib telefon, hammaga tavsiya qilaman",
    "Taom mazali, xizmat zoʻr",
    "Kitob qiziqarli, oʻqishni tavsiya qilaman",
    "Yaxshi restoran, hammasi juda mazali",
    "Quloqchinlar yaxshi, ovozi ajoyib",
    "Zerikarli emas, kino yoqdi",
    "Narxi qimmat emas, sifati yaxshi",
    "Yetkazib berish tez, hammasi yoqdi, rahmat",
    "Afsuslanmadim, ajoyib xarid",
    "Xodimlar xushmuomala, xona toza, tavsiya qilaman",
    "Juda mamnunman, telefon tez ishlaydi",
    "Syujet qiziqarli, aktyorlar yaxshi",
    "Palov mazali, musiqa yoqimli",
    "Kitob yoqdi, oson oʻqiladi",
    "Yomon emas, hammaga tavsiya qilaman",
    "Zoʻr kafe, mazali va arzon",
    "Kamera yaxshi, xariddan juda mamnunman",
    "Aktyorlar zoʻr oʻynagan, tavsiya qilaman",
    "Xizmat tez va xushmuomala",
    "Qulay va chiroyli, tavsiya qilaman",
    "Butun oila uchun yaxshi kino",
    "Hammasi yoqdi, yana kelamiz",
    "Sifati ajoyib, sotuvchiga rahmat",
    # test
    "Ajoyib kino, juda yoqdi",
    "Mazali, tez va arzon, tavsiya qilaman",
    "Zerikarli emas, qiziqarli kitob",
    "Sifati yaxshi, afsuslanmadim",
    "Xodimlar xushmuomala, hammasi yoqdi",
    "Yomon emas, zoʻr telefon",
]
neg = [
    "Kino umuman yoqmadi, syujeti zerikarli",
    "Yomon telefon, hech kimga tavsiya qilmayman",
    "Taom sovuq, xizmat yomon",
    "Kitob zerikarli, oʻqishni tavsiya qilmayman",
    "Yomon restoran, hech narsa mazali emas",
    "Quloqchinlar yomon, ovozi xira",
    "Juda zerikarli, kino yoqmadi",
    "Narxi qimmat, sifati yaxshi emas",
    "Yetkazib berish sekin, hech narsa yoqmadi",
    "Afsuslandim, yomon xarid",
    "Xodimlar qoʻpol, xona iflos, tavsiya qilmayman",
    "Juda afsusdaman, telefon bir haftada buzildi",
    "Syujet zerikarli, aktyorlar yaxshi emas",
    "Palov sovuq, musiqa baland",
    "Kitob yoqmadi, qiyin oʻqiladi",
    "Yaxshi emas, hech kimga tavsiya qilmayman",
    "Yomon kafe, mazasiz va qimmat",
    "Kamera yomon, xariddan juda afsusdaman",
    "Aktyorlar yomon oʻynagan, tavsiya qilmayman",
    "Xizmat sekin va qoʻpol",
    "Noqulay va ogʻir, tavsiya qilmayman",
    "Yomon kino, vaqtimga achinaman",
    "Hech narsa yoqmadi, boshqa kelmaymiz",
    "Sifati yomon, pulimni qaytarishmadi",
    # test
    "Yomon kino, umuman yoqmadi",
    "Mazali emas, sekin va qimmat, tavsiya qilmayman",
    "Juda zerikarli, qiziqarli emas",
    "Sifati yaxshi emas, afsuslandim",
    "Xodimlar qoʻpol, hech narsa yoqmadi",
    "Mazasi yoʻq, yomon kafe",
]

def tokenize(text):
    return re.findall(r"\w+", text.lower())

# har bir sinfning oxirgi 6 ta sharhi — test
train = [(t, "+") for t in pos[:24]] + [(t, "-") for t in neg[:24]]
test = [(t, "+") for t in pos[24:]] + [(t, "-") for t in neg[24:]]
X_train, y_train = [t for t, _ in train], np.array([c for _, c in train])
X_test, y_test = [t for t, _ in test], np.array([c for _, c in test])
print(f"oʻqitish: {len(train)} ({(y_train == '+').sum()} +, {(y_train == '-').sum()} −);  test: {len(test)}")
lens = [len(tokenize(t)) for t in pos + neg]
print(f"sharhdagi soʻzlar: eng kami {min(lens)} eng koʻpi {max(lens)}, oʻrtacha {np.mean(lens):.1f}")
print("\nMisollar:")
for t, c in train[:3] + train[24:27]:
    print(f"  {c}  {t}")

## 2. Multinomial sodda Bayes noldan

$d$ sharh — $w_1,\dots,w_n$ soʻzlar qopi. Bayes formulasi va soʻzlar mustaqilligi haqidagi «sodda» farazga koʻra

$$\hat c=\arg\max_c\Big[\log P(c)+\sum_{i=1}^{n}\log P(w_i\mid c)\Big],\qquad P(w\mid c)=\frac{\text{count}(w,c)+\alpha}{\sum_{v}\text{count}(v,c)+\alpha|V|}.$$

$\alpha=1$ — Laplas silliqlashi: usiz sinfda uchramagan soʻz $P=0$ va $\log P=-\infty$ berardi, bitta shunday «nol» butun sharhni yoʻqqa chiqarardi. Ehtimollarni koʻpaytirish oʻrniga logarifmlarni qoʻshamiz: yuzlab kichik sonlar koʻpaytmasi mashina noliga aylanib qoladi.

In [ ]:
from collections import Counter

def train_nb(docs, labels, alpha=1.0, featurize=tokenize):
    classes = sorted(set(labels))
    counts = {c: Counter() for c in classes}
    n_docs = Counter(labels)
    for d, c in zip(docs, labels):
        counts[c].update(featurize(d))
    vocab = sorted(set(w for c in classes for w in counts[c]))
    log_prior = {c: np.log(n_docs[c] / len(docs)) for c in classes}
    log_lik = {}
    for c in classes:
        total = sum(counts[c].values())
        log_lik[c] = {w: np.log((counts[c][w] + alpha) / (total + alpha * len(vocab))) for w in vocab}
    return dict(classes=classes, vocab=set(vocab), log_prior=log_prior, log_lik=log_lik,
                counts=counts, featurize=featurize)

def nb_scores(model, text):
    words = [w for w in model["featurize"](text) if w in model["vocab"]]   # notanish soʻzlarni tashlab ketamiz
    return {c: model["log_prior"][c] + sum(model["log_lik"][c][w] for w in words) for c in model["classes"]}

def nb_predict(model, text):
    s = nb_scores(model, text)
    return max(s, key=s.get)

nb = train_nb(X_train, y_train)
print("lugʻat:", len(nb["vocab"]), "soʻz;  log P(+) =", round(nb["log_prior"]["+"], 3), " log P(−) =", round(nb["log_prior"]["-"], 3))
for c in nb["classes"]:
    print(f"  sinfdagi soʻzlar {c}: {sum(nb['counts'][c].values())},  eng koʻp uchraydiganlar:", nb["counts"][c].most_common(6))

print("\nBir nechta soʻz uchun haqiqatga oʻxshashlik jadvali (α = 1):")
rows = []
for w in ["tavsiya", "qilaman", "qilmayman", "yoqdi", "yoqmadi", "yaxshi", "yomon", "emas"]:
    rows.append([w, nb["counts"]["+"][w], nb["counts"]["-"][w], np.exp(nb["log_lik"]["+"][w]), np.exp(nb["log_lik"]["-"][w])])
print(pd.DataFrame(rows, columns=["soʻz", "count(+)", "count(−)", "P(w|+)", "P(w|−)"]).round(4).to_string(index=False))

acc = np.mean([nb_predict(nb, t) == c for t, c in test])
print(f"\ntestdagi accuracy: {acc:.2f}")

**Model nega shunday qaror qildi?** Ikki sinf uchun logarifmlar ayirmasiga qarash qulay: $w$ soʻzining hissasi $\log P(w\mid +)-\log P(w\mid -)$ ga teng. Musbat hissa «+» tomonga, manfiysi «−» tomonga tortadi. Hissalar yigʻindisi va apriori logarifmlar ayirmasi — imkoniyatlar nisbatining logarifmi, aposterior ehtimol $P(+\mid d)=\sigma(\text{yigʻindi})$.

In [ ]:
def explain(model, text, plot=False):
    s = nb_scores(model, text)
    prior = model["log_prior"]["+"] - model["log_prior"]["-"]
    rows = []
    for w in model["featurize"](text):
        if w in model["vocab"]:
            rows.append((w, model["log_lik"]["+"][w] - model["log_lik"]["-"][w]))
        else:
            rows.append((w, None))
    total = prior + sum(v for _, v in rows if v is not None)
    p_pos = 1 / (1 + np.exp(-total))
    print(text)
    for w, v in rows:
        print(f"   {w:<16}", "lugʻatda yoʻq" if v is None else f"{v:+.2f}")
    print(f"   {'apriori':<16} {prior:+.2f}\n   {'yigʻindi':<16} {total:+.2f}  →  P(+|d) = {p_pos:.3f}  →  {max(s, key=s.get)}")
    if plot:
        known = [(w, v) for w, v in rows if v is not None]
        plt.figure(figsize=(7, 2.8))
        plt.barh([w for w, _ in known][::-1], [v for _, v in known][::-1],
                 color=["#17907E" if v > 0 else "#D9532C" for _, v in known][::-1])
        plt.axvline(0, color="black", lw=0.8)
        plt.xlabel("log P(w|+) − log P(w|−)")
        plt.title("Soʻzlar hissasi: yashillar «+» ga, qizillar «−» ga tortadi")
        plt.tight_layout()
        plt.show()

explain(nb, "Xizmat yaxshi emas, taom ham mazali emas", plot=True)
print()
explain(nb, "Kino zerikarli emas, aktyorlar yaxshi")

scikit-learn bilan solishtiramiz: `MultinomialNB(alpha=1.0)` xuddi shu kattaliklarni `class_log_prior_` va `feature_log_prob_` da saqlaydi.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
Xtr = cv.fit_transform(X_train)
sk_nb = MultinomialNB(alpha=1.0).fit(Xtr, y_train)

vocab_sk = cv.get_feature_names_out()
ours = np.array([[nb["log_lik"][c][w] for w in vocab_sk] for c in sk_nb.classes_])
print("sinflar:", sk_nb.classes_)
print("apriori mos keladi:    ", np.allclose(sk_nb.class_log_prior_, [nb["log_prior"][c] for c in sk_nb.classes_]))
print("oʻxshashliklar mos keladi:", np.allclose(sk_nb.feature_log_prob_, ours))
pred_sk = sk_nb.predict(cv.transform(X_test))
pred_ours = np.array([nb_predict(nb, t) for t in X_test])
print("testdagi bashoratlar mos keladi:", np.array_equal(pred_sk, pred_ours))
for t, c, p in zip(X_test, y_test, pred_sk):
    print(f"   {'✓' if c == p else '✗'} haqiqat {c} model {p}  {t}")

## 3. Inkor

Oʻzbek tilida inkor ikki xil ifodalanadi, va bu soʻzlar qopi uchun muhim:

* **feʼldagi qoʻshimcha** `-ma-`: «yoqdi» → «yoqmadi», «tavsiya qilaman» → «tavsiya qilmayman». Inkor soʻzning ichida, shuning uchun BoW uchun «yoqmadi» — «yoqdi» dan alohida soʻz va oʻz statistikasiga ega. Rus tilidagi «не понравился» dan farqli oʻlaroq, bu yerda qoʻshimcha ishlov kerak emas (lekin har bir inkor shakl oʻqitish maʼlumotlarida uchrashi kerak).
* **analitik inkor** `emas` (va «yoʻq»): «yaxshi emas», «qimmat emas», «zerikarli emas». Bu yerda BoW «yaxshi» ni «emas» dan alohida koʻradi, «emas» esa ikkala sinfda uchraydi («yaxshi emas», lekin «yomon emas» ham).

Yuqoridagi birinchi misolga qarang: «Xizmat yaxshi emas, taom ham mazali emas» ijobiy deb topildi ($P(+\mid d)=0{,}77$), chunki «yaxshi» (+0,58) va «mazali» (+0,94) oʻz hissasini qoʻshadi, «emas» esa atigi −0,20 beradi.

Maʼlum usul (Das & Chen, 2001; Pang et al., 2002): inkorni soʻzga yopishtirish. «yaxshi emas» → `EMAS_yaxshi`, «mazasi yoʻq» → `YOʻQ_mazasi`. Oʻzbek tilida inkor soʻzi **oldingi** soʻzga tegishli, rus tilida esa «не» **keyingi** soʻzga.

In [ ]:
NEG_WORDS = {"emas", "yoʻq"}

def mark_negation(text):
    # «yaxshi emas» → «EMAS_yaxshi»: inkor soʻzi oldingi soʻzga yopishtiriladi
    out = []
    for w in tokenize(text):
        if w in NEG_WORDS and out:
            out[-1] = w.upper() + "_" + out[-1]
        else:
            out.append(w)
    return out

for t in ["Xizmat yaxshi emas, taom ham mazali emas"] + [t for t in pos + neg if "yoqdi" in tokenize(t)][:2]:
    print(f"{t}\n   → {mark_negation(t)}")

nb_neg = train_nb(X_train, y_train, featurize=mark_negation)
print("\nlugʻatdagi yangi inkorli belgilar:", sorted(w for w in nb_neg["vocab"] if "_" in w))

tricky = [
    ("Taom mazali emas", "-"),
    ("Kino zerikarli emas", "+"),
    ("Telefon yaxshi emas", "-"),
    ("Narxi qimmat emas", "+"),
    ("Xizmat yoqmadi", "-"),
]
print(f"\n{'':<36} NB     NB+inkor")
for t, gold in tricky:
    print(f"{t:<36} {nb_predict(nb, t):<6} {nb_predict(nb_neg, t):<6} (toʻgʻrisi: {gold})")
acc_neg = np.mean([nb_predict(nb_neg, t) == c for t, c in test])
print(f"\ntestdagi accuracy: NB {acc:.2f},  NB+inkor {acc_neg:.2f}")
print()
explain(nb_neg, "Xizmat yaxshi emas, taom ham mazali emas")

## 4. Chiziqli SVM: hinge loss boʻyicha gradient tushish

SVM ham chiziqli model: $\text{score}(d)=w\cdot x_d+b$, sinf — score ishorasi. Lekin vaznlar boshqacha tanlanadi: belgi $y\in\{+1,-1\}$, va biz har bir misol shunchaki kerakli tomonda emas, kamida 1 ga teng **chegara** (margin) bilan yotishini xohlaymiz: $y(w\cdot x+b)\ge 1$. Yoʻqotish funksiyasi

$$L(w,b)=\frac{\lambda}{2}\|w\|^2+\frac1N\sum_{i}\max\big(0,\;1-y_i(w\cdot x_i+b)\big).$$

Chegarasi 1 dan katta misollar gradientga taʼsir qilmaydi (ular «uzoqda»), faqat buzuvchilardan oʻrganiladi. Belgilar — inkorli TF-IDF. Toʻliq gradient tushish bilan oʻqitamiz: har bir epoxa — 48 ta oʻquv sharhi boʻyicha bitta qadam.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(tokenizer=mark_negation, lowercase=False, token_pattern=None)
Xtr_tf = tf.fit_transform(X_train).toarray()
Xte_tf = tf.transform(X_test).toarray()
ytr = np.where(y_train == "+", 1, -1)
yte = np.where(y_test == "+", 1, -1)

def train_svm(X, y, lam=0.01, lr=0.5, epochs=300):
    N, D = X.shape
    w, b = np.zeros(D), 0.0
    hist = []
    for ep in range(epochs):
        margins = y * (X @ w + b)
        viol = margins < 1                                   # chegarani buzuvchilar
        loss = lam / 2 * w @ w + np.maximum(0, 1 - margins).mean()
        hist.append((loss, (margins > 0).mean(), viol.sum()))
        grad_w = lam * w - (y[viol, None] * X[viol]).sum(axis=0) / N
        grad_b = -y[viol].sum() / N
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b, np.array(hist)

w_svm, b_svm, hist = train_svm(Xtr_tf, ytr)
for ep in (0, 10, 50, 100, 299):
    print(f"epoxa {ep:3d}: yoʻqotish {hist[ep, 0]:.3f}, oʻqitishdagi accuracy {hist[ep, 1]:.2f}, chegarani buzuvchilar {int(hist[ep, 2])}")
acc_svm = np.mean(np.sign(Xte_tf @ w_svm + b_svm) == yte)
print(f"\ntestdagi accuracy (bizning SVM): {acc_svm:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist[:, 0], color="#2B55C9")
ax[0].set_title("Yoʻqotish (hinge + L2)")
ax[0].set_xlabel("epoxa")
ax[1].plot(hist[:, 1], color="#17907E", label="oʻqitishdagi accuracy")
ax[1].plot(hist[:, 2] / len(ytr), color="#D9532C", label="chegarani buzuvchilar ulushi")
ax[1].set_xlabel("epoxa")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Kutubxona modellari bilan solishtiramiz: `LinearSVC(loss="hinge")` xuddi shu masalani yechadi ($C$ parametri $1/(\lambda N)$ ga mos keladi), `LogisticRegression` — xuddi shu chiziqli model, lekin logistik yoʻqotish funksiyasi bilan. Eng kuchli soʻzlarni koʻramiz: SVM da bu $w$ vaznlari, NB da — logarifmlar ayirmasi.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

N = Xtr_tf.shape[0]
svc = LinearSVC(loss="hinge", C=1 / (0.01 * N), max_iter=20000, random_state=0).fit(Xtr_tf, y_train)
logreg = LogisticRegression(C=10, max_iter=1000).fit(Xtr_tf, y_train)
w_sk = svc.coef_[0] if svc.classes_[1] == "+" else -svc.coef_[0]
print("vaznlar korrelyatsiyasi: bizning SVM va LinearSVC:", round(float(np.corrcoef(w_svm, w_sk)[0, 1]), 3))
for name, pred in [("bizning SVM", np.where(Xte_tf @ w_svm + b_svm > 0, "+", "-")),
                   ("LinearSVC", svc.predict(Xte_tf)), ("LogisticRegression", logreg.predict(Xte_tf))]:
    print(f"   {name:<20} testdagi accuracy {np.mean(pred == y_test):.2f}")

vocab_tf = tf.get_feature_names_out()
order = np.argsort(w_svm)
nb_ratio = {w: nb_neg["log_lik"]["+"][w] - nb_neg["log_lik"]["-"][w] for w in nb_neg["vocab"]}
top_nb = sorted(nb_ratio, key=nb_ratio.get)
print("\nSVM  «+»:", ", ".join(vocab_tf[order[::-1][:8]]))
print("SVM  «−»:", ", ".join(vocab_tf[order[:8]]))
print("NB   «+»:", ", ".join(top_nb[::-1][:8]))
print("NB   «−»:", ", ".join(top_nb[:8]))

top = np.concatenate([order[:8], order[-8:]])
plt.figure(figsize=(7, 4.6))
plt.barh(vocab_tf[top], w_svm[top], color=["#D9532C"] * 8 + ["#17907E"] * 8)
plt.axvline(0, color="black", lw=0.8)
plt.title("SVM ning eng kuchli vaznlari")
plt.tight_layout()
plt.show()

## 5. Sifatni baholash

12 ta test sharhida modellarni **xatolar matritsasi** (confusion matrix) va metrikalar boʻyicha solishtiramiz: *aniqlik* (precision) — model ijobiy degan sharhlarning qancha qismi haqiqatan ijobiy; *toʻliqlik* (recall) — model ijobiy sharhlarning qancha qismini topdi; F1 — ularning garmonik oʻrtachasi.

Inkorsiz NB testda xato qiladi, shuning uchun batafsil hisobotni u uchun chiqaramiz. 12 ta sharh juda kam, bitta xato accuracy ni 8 foiz punktga oʻzgartiradi. **Kross-validatsiya** ishonchliroq: 60 ta sharhning hammasini 5 qismga boʻlamiz, 5 marta toʻrttasida oʻqitib, beshinchisida tekshiramiz. Halol solishtirish uchun lugʻat va IDF har bir boʻlinish ichida qaytadan quriladi (`Pipeline`).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

models = {
    "NB": pred_ours,
    "NB+inkor": np.array([nb_predict(nb_neg, t) for t in X_test]),
    "SVM": svc.predict(Xte_tf),
    "LogReg": logreg.predict(Xte_tf),
}
labels = ["+", "-"]
fig, axes = plt.subplots(1, len(models), figsize=(12, 3))
for ax, (name, pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, pred, labels=labels)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=6)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=14, color="white" if cm[r, c] > 3 else "black")
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("bashorat")
    ax.set_title(f"{name}: {np.mean(pred == y_test):.2f}")
axes[0].set_ylabel("aslida")
plt.tight_layout()
plt.show()

print("Inkorsiz NB:")
print(classification_report(y_test, models["NB"], digits=2, zero_division=0))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_all = pos + neg
y_all = np.array(["+"] * len(pos) + ["-"] * len(neg))

def vec(neg_marking, tfidf):
    V = TfidfVectorizer if tfidf else CountVectorizer
    return V(tokenizer=mark_negation if neg_marking else tokenize, lowercase=False, token_pattern=None)

pipelines = {
    "NB": make_pipeline(vec(False, False), MultinomialNB()),
    "NB+inkor": make_pipeline(vec(True, False), MultinomialNB()),
    "SVM": make_pipeline(vec(False, True), LinearSVC(C=1.0, random_state=0)),
    "SVM+inkor": make_pipeline(vec(True, True), LinearSVC(C=1.0, random_state=0)),
    "LogReg+inkor": make_pipeline(vec(True, True), LogisticRegression(C=10, max_iter=1000)),
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res = {}
for name, pipe in pipelines.items():
    sc = cross_val_score(pipe, X_all, y_all, cv=cv5)
    res[name] = sc
    print(f"{name:<18} {sc.mean():.2f} ± {sc.std():.2f}   {np.round(sc, 2)}")

plt.figure(figsize=(8, 3.2))
plt.bar(list(res), [s.mean() for s in res.values()], yerr=[s.std() for s in res.values()],
        color="#2B55C9", capsize=4)
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.title("Kross-validatsiya, 5 boʻlinish (oʻrtacha ± st. ogʻish)")
plt.tight_layout()
plt.show()

Inkor NB uchun ham, SVM uchun ham taxminan 17 foiz punkt qoʻshadi, va bu faqat oʻrtacha emas, deyarli har bir boʻlinishda koʻrinadi. Bir xil belgilardagi NB va SVM orasidagi farq esa (2 punkt) standart ogʻishdan kichik: 60 ta sharhda bu shovqin. Bunday kichik korpusda belgilar klassifikator tanlovidan muhimroq; haqiqiy xulosalar uchun yuzlab va minglab belgilangan sharhlar kerak.

## 6. Topshiriqlar

1. **Silliqlash.** `train_nb` ni $\alpha = 0{,}01$, $0{,}1$, $1$, $5$ bilan oʻqiting va testdagi accuracy hamda «yoqdi» soʻzining hissasini chiqaring. Nega kichik $\alpha$ da kam uchraydigan soʻzlarning hissasi juda katta boʻlib ketadi?
2. **Feʼl inkori.** `mark_negation` ga qoida qoʻshing: `-madi`, `-maydi`, `-mayman` bilan tugaydigan soʻzlar uchun soʻzning oʻzidan tashqari umumiy `NEG` belgisini ham qoʻshing. NB kross-validatsiyasi yaxshilanadimi? Nega bu belgi kam yangi maʼlumot berishi mumkin («yoqdi» va «yoqmadi» allaqachon turli soʻzlar)? Oʻqitishda uchramagan inkor feʼl uchun u qachon foydali boʻladi?
3. **Oʻz sharhlaringiz.** NB+inkor modeli adashadigan 5 ta sharh yozing (masalan, kinoya bilan: «Ha, zoʻr film ekan, 10 daqiqada uxlab qoldim»). Qaysi soʻzlar notoʻgʻri hissa beradi? Oʻqitishga 2–3 ta sharh qoʻshib, buni tuzatish mumkinmi?
4. **Bigrammalar.** 5-boʻlimdagi `TfidfVectorizer(...)` ni `ngram_range=(1, 2)` varianti bilan almashtiring (`mark_negation` siz). `EMAS_` orqali inkorli SVM va bigrammali SVM kross-validatsiyasini solishtiring: qaysi usul «yaxshi emas» ni yaxshiroq ushlaydi?
5. **Neytral sinf.** 10 ta neytral sharh qoʻshing (`"0"` belgisi, masalan «Filmni shanba kuni koʻrdim») va `MultinomialNB` hamda `LinearSVC` ni uch sinfda oʻqiting. 3 × 3 xatolar matritsasini quring. Qaysi sinflar koʻproq adashtiriladi?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing